In [ ]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import json, glob, os
from PIL import Image


# Path to one of your spatial binning levels
base_dir = "./Prostate_spatial_data"

# finding unique samplle prefixes
samples = sorted(set(
    "_".join(os.path.basename(f).split("_")[:3])
    for f in glob.glob(os.path.join(base_dir, "*_matrix.mtx"))
))

print(samples)

adatas = {}

for sample in samples:
    print(f"Loading {sample} ...")
    prefix = os.path.join(base_dir, sample)
    
    # Read 10x count matrix
    ad = sc.read_mtx(prefix + "_matrix.mtx").T
    ad.var_names = pd.read_csv(prefix + "_features.tsv", sep="\t", header=None)[1]
    ad.obs_names = pd.read_csv(prefix + "_barcodes.tsv", sep="\t", header=None)[0]
    
    #  Load spatial coordinates 
    pos = pd.read_csv(prefix + "_tissue_positions_list.csv", header=None)
    pos.columns = ["barcode", "in_tissue", "array_row", "array_col", "pxl_row_in_fullres", "pxl_col_in_fullres"]
    pos.index = pos["barcode"]

    #  keep only barcodes that exist in ad.obs_names
    pos = pos.loc[pos["barcode"].isin(ad.obs_names), :]

    # ensure ordering matches AnnData obs
    pos = pos.reindex(ad.obs_names)

    # Join metadata and assign spatial coordinates
    ad.obs = ad.obs.join(pos, how="left")
    ad.obsm["spatial"] = pos[["pxl_row_in_fullres", "pxl_col_in_fullres"]].to_numpy(dtype=float)

    # --- Load scalefactors ---
    with open(prefix + "_scalefactors_json.json") as f:
        ad.uns["spatial"] = {sample: {"scalefactors": json.load(f)}}
    
    # Load images 
    img = Image.open(prefix + "_tissue_lowres_image.png")
    ad.uns["spatial"][sample]["images"] = {"lowres": np.array(img)}
    
    ad.uns["spatial"][sample]["metadata"] = {"source": sample}
    
    adatas[sample] = ad

print(f"\nLoaded {len(adatas)} spatial samples.")
